# 04 — Exploration & Visualisations

Notebook d'exploration des données silver.
Inclut un **avant/après** : un graphique naïf puis sa version corrigée,
avec justification par les principes de perception visuelle.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
from deltalake import DeltaTable
from pathlib import Path

DATA_DIR = Path("../data")
SILVER_PATH = str(DATA_DIR / "silver" / "flights")

df = DeltaTable(SILVER_PATH).to_pandas()
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
df["year"] = df["FL_DATE"].dt.year
df["month"] = df["FL_DATE"].dt.month
df["day_of_week"] = df["FL_DATE"].dt.dayofweek

print(f"Dataset : {len(df):,} lignes")
df.head()

## 1. Distribution des retards

In [ ]:
delays = df[df["ARR_DELAY"].notna() & (df["ARR_DELAY"] > -30) & (df["ARR_DELAY"] < 180)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(delays["ARR_DELAY"], bins=100, color="#3D7ADB", edgecolor="white", linewidth=0.3)
ax.axvline(x=0, color="#333", linewidth=1, linestyle="--", label="À l'heure")
ax.axvline(x=15, color="#D94E72", linewidth=1, linestyle="--", label="Seuil retard (15 min)")
ax.set_xlabel("Retard à l'arrivée (minutes)")
ax.set_ylabel("Nombre de vols")
ax.set_title("Distribution des retards à l'arrivée")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
plt.tight_layout()
plt.show()

## 2. Retard moyen par jour de la semaine

In [ ]:
JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
by_dow = df.groupby("day_of_week")["ARR_DELAY"].mean().reset_index()
by_dow["jour"] = by_dow["day_of_week"].map(lambda x: JOURS[x])

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(by_dow["jour"], by_dow["ARR_DELAY"], color="#2FAC91", width=0.6)
ax.set_ylabel("Retard moyen (min)")
ax.set_title("Retard moyen à l'arrivée par jour de la semaine")
ax.spines[["top", "right"]].set_visible(False)
ax.set_axisbelow(True)
ax.yaxis.grid(True, color="#E8E8E8")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Top 10 aéroports les plus en retard

In [ ]:
top_airports = (
    df.groupby("ORIGIN")
    .agg(avg_delay=("DEP_DELAY", "mean"), flights=("FL_DATE", "count"))
    .query("flights > 10000")
    .nlargest(10, "avg_delay")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_airports["ORIGIN"], top_airports["avg_delay"], color="#D94E72", height=0.6)
ax.set_xlabel("Retard moyen au départ (min)")
ax.set_title("Top 10 aéroports — Retard moyen au départ")
ax.spines[["top", "right"]].set_visible(False)
ax.invert_yaxis()
ax.set_axisbelow(True)
ax.xaxis.grid(True, color="#E8E8E8")
for i, v in enumerate(top_airports["avg_delay"]):
    ax.text(v + 0.3, i, f"{v:.1f} min", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Évolution mensuelle — Taux de ponctualité

In [ ]:
monthly = df.groupby(["year", "month"]).agg(
    total=("FL_DATE", "count"),
    on_time=("ARR_DEL15", lambda x: (x == 0).sum()),
).reset_index()
monthly["on_time_pct"] = 100 * monthly["on_time"] / monthly["total"]
monthly["period"] = monthly.apply(lambda r: f"{int(r['year'])}-{int(r['month']):02d}", axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly["period"], monthly["on_time_pct"], marker="o", color="#3D7ADB",
        linewidth=2, markersize=5)
ax.set_ylabel("Ponctualité (%)")
ax.set_title("Taux de ponctualité mensuel")
ax.set_ylim(50, 100)
ax.spines[["top", "right"]].set_visible(False)
ax.set_axisbelow(True)
ax.yaxis.grid(True, color="#E8E8E8")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---

## 5. Avant / Après : correction d'un graphique

### AVANT — Version naïve

Un camembert (pie chart) pour montrer les causes de retard.

**Problèmes** :
- Le pie chart rend la comparaison de proportions proches très difficile
  (principe de perception : l'œil humain compare mal les angles).
- Les couleurs sont aléatoires (palette par défaut de matplotlib), pas
  ordonnées par valeur.
- Les labels se chevauchent quand les parts sont petites.
- Pas de titre explicite.

In [ ]:
# AVANT — Pie chart naïf
delay_cols = ["CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
              "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]
cause_totals = df[delay_cols].sum()
cause_totals.index = ["Compagnie", "Météo", "Contrôle aérien", "Sécurité", "Avion en retard"]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(cause_totals, labels=cause_totals.index, autopct="%1.1f%%")
ax.set_title("Causes de retard")
plt.show()

### APRÈS — Version corrigée

Un bar chart horizontal, trié par valeur décroissante.

**Corrections appliquées** (principes de perception) :

1. **Forme → bar chart horizontal** : l'œil compare les longueurs bien mieux que
   les angles (loi de Stevens : exposant ~1.0 pour la longueur vs ~0.7 pour l'aire).
2. **Tri décroissant** : facilite la lecture et le classement immédiat
   (principe de pré-attention : l'ordre spatial guide le regard).
3. **Palette à une seule teinte** (séquentielle) : la couleur encode la magnitude,
   du plus foncé (plus grand) au plus clair. Pas de couleurs arbitraires.
4. **Labels directs** sur les barres : évite le va-et-vient avec une légende
   (réduction de la charge cognitive).
5. **Grille légère** et axes épurés (encre minimale, ratio données/encre de Tufte).

In [ ]:
# APRÈS — Bar chart corrigé
cause_sorted = cause_totals.sort_values(ascending=True)
cause_pct = 100 * cause_sorted / cause_sorted.sum()

n = len(cause_pct)
blues = [f"rgba(61, 122, 219, {0.3 + 0.7 * i / (n - 1)})" for i in range(n)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(cause_pct.index, cause_pct.values, color=["#A8C6F0", "#7BAAE0", "#5B93D6", "#3D7ADB", "#2560B5"],
               height=0.6)
ax.set_xlabel("Part des retards (%)")
ax.set_title("Causes de retard — Répartition", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.set_axisbelow(True)
ax.xaxis.grid(True, color="#E8E8E8")

for bar, val in zip(bars, cause_pct.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{val:.1f} %", va="center", fontsize=10)

plt.tight_layout()
plt.show()

### Résumé des corrections

| Aspect | Avant (naïf) | Après (corrigé) | Principe de perception |
|--------|-------------|-----------------|------------------------|
| Forme | Pie chart | Bar chart horizontal | Longueur > Angle (Stevens) |
| Ordre | Arbitraire | Trié décroissant | Pré-attention spatiale |
| Couleur | Palette par défaut (catégorielle) | Séquentielle mono-teinte | Magnitude = luminosité |
| Labels | Sur le pie (chevauchement) | Directement sur les barres | Charge cognitive réduite |
| Grille | Absente | Légère, en arrière-plan | Ratio données/encre (Tufte) |